This notebook trains the two classifiers for the rice mill prototype and measures how
accurate they actually are.

Model 1 grades a rice grain as Whole, Stained, Broken or Chalky. That is what a mill prices on.

Model 2 identifies disease on a rice leaf, for the farmers who supply the mill.

Both are image classifiers built on YOLOv11. They are trained separately, on separate data.
Putting them in one model would make both worse, because the model would learn to tell a green
leaf photo from a white grain photo, which is trivial, instead of learning chalky from whole,
which is the hard part.

The point of this notebook is an honest accuracy number. Three things make it honest rather
than flattering.

The data is split three ways, not two. The model learns from the training split. The
validation split decides when to stop training. The test split is held back and looked at once
at the very end. Only the test number can be quoted, because the other two influenced the
model.

Duplicate images are removed before splitting. Several public rice datasets are re-uploads of
each other. If the same photo lands in both training and test, the model gets marked on
questions it has already seen, and the score means nothing.

Accuracy is reported per class, not just overall. A model that never predicts Chalky can still
score well overall if chalky grains are rare. The confusion matrix shows that; a single number
hides it.

Check the GPU before anything else.

The cell below lists what Colab has attached. If it prints nothing useful, the runtime has no
GPU and training would take days instead of hours. Go to Runtime, then Change runtime type,
and select T4 GPU.

There is a second check after the install that stops the notebook outright if no GPU is
available. It runs after the install rather than before, because importing torch before pip
has finished can leave the session holding a half replaced library.

In [ ]:
!nvidia-smi

Install the libraries, then confirm the GPU is really there.

If a later import fails, Colab has swapped a torch version underneath the install. Use
Runtime, then Restart session, and carry on from the import cell. Do not run the install
again.

In [ ]:
!pip install ultralytics kagglehub scikit-learn pillow matplotlib -q

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run."
)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
import collections
import hashlib
import json
import os
import random
import shutil
from pathlib import Path

import numpy as np
from PIL import Image
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

WORK = Path("/content/work")
OUT = Path("/content/artifacts")
for d in (WORK, OUT):
    d.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
print("imports OK")

Kaggle credentials.

The datasets come from Kaggle, which needs an API key. Without one the download fails with a
401 that reads like a network problem.

In the Colab sidebar open Secrets, add KAGGLE_USERNAME and KAGGLE_KEY from your Kaggle account
settings, and enable notebook access for both. The names must match exactly.

In [ ]:
import kagglehub

try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Kaggle credentials loaded from Colab secrets.")
except Exception as e:
    print("Could not load Colab secrets:", e)
    print("Falling back to any kaggle.json already present. If downloads 401, fix this first.")

Step 0: Download the datasets and look at what actually arrived

Each source is downloaded on its own and reported separately. A source that fails to download
is skipped with a message rather than stopping the notebook, so a bad or renamed slug does not
cost you the whole run.

The grain model uses one source. The leaf model merges several, which is legitimate because
they answer the same question. Merging the grain data and the leaf data would not be.

Check the printed counts before going further. They decide which classes are worth keeping.

In [ ]:
GRAIN_SOURCES = [
    ("peru_mill", "cristhiansempertegui/dataset-de-arroz-peruano"),
]

# Several of these are re-uploads of each other. That is handled by the duplicate pass
# further down, not by trusting the descriptions.
LEAF_SOURCES = [
    ("philippines", "shrupyag001/philippines-rice-diseases"),
    ("sethy", "nirmalsankalana/rice-leaf-disease-image"),
    ("dedeikhsan", "dedeikhsandwisaputra/rice-leafs-disease-dataset"),
]


def download(sources):
    got = {}
    for name, slug in sources:
        try:
            path = kagglehub.dataset_download(slug)
            got[name] = Path(path)
            print("ok      %-14s %s" % (name, path))
        except Exception as e:
            print("FAILED  %-14s %s: %s" % (name, slug, type(e).__name__))
    return got


print("grain sources")
grain_paths = download(GRAIN_SOURCES)
print()
print("leaf sources")
leaf_paths = download(LEAF_SOURCES)

assert grain_paths, "no grain dataset downloaded, cannot continue"
assert leaf_paths, "no leaf dataset downloaded, cannot continue" 

Find the folder that holds the class subfolders, for each source.

The layout differs between datasets and none of them guarantee where the images sit. Guessing
a path wrong is not loud: the label for each image comes from the name of its parent folder,
so a root at the wrong level produces labels that are quietly meaningless.

This looks for the directory whose subfolders actually contain images, and prints what it
found so it can be checked by eye.

In [ ]:
def find_class_root(root, min_classes=2):
    """Deepest directory whose subfolders directly contain images."""
    best, best_count = None, 0
    for d in [root, *root.rglob("*")]:
        if not d.is_dir():
            continue
        subs = [s for s in d.iterdir() if s.is_dir()]
        if len(subs) < min_classes:
            continue
        ok = [s for s in subs
              if any(f.suffix.lower() in IMAGE_EXTS for f in s.iterdir() if f.is_file())]
        if len(ok) >= min_classes and len(ok) > best_count:
            best, best_count = d, len(ok)
    return best


def inventory(paths):
    found = {}
    for name, path in paths.items():
        root = find_class_root(path)
        if root is None:
            print("%s: no class-folder layout found, skipping" % name)
            continue
        counts = {}
        for sub in sorted(d for d in root.iterdir() if d.is_dir()):
            n = sum(1 for f in sub.rglob("*") if f.suffix.lower() in IMAGE_EXTS)
            if n:
                counts[sub.name] = n
        found[name] = (root, counts)
        print("%s  ->  %s" % (name, root))
        for k, v in sorted(counts.items(), key=lambda kv: -kv[1]):
            print("    %-34s %5d" % (k, v))
        print("    %-34s %5d" % ("TOTAL", sum(counts.values())))
        print()
    return found


print("GRAIN")
grain_inv = inventory(grain_paths)
print("LEAF")
leaf_inv = inventory(leaf_paths)

Confirm the model weights download.

The Ultralytics documentation now defaults to a newer generation than YOLOv11, so check the
weights we actually want are still available. This takes seconds and saves finding out an hour
into a training run.

In [ ]:
for w in ("yolo11n-cls.pt", "yolo11n.pt"):
    try:
        YOLO(w)
        print("ok     ", w)
    except Exception as e:
        print("FAILED ", w, type(e).__name__, e)

Licences.

This is going into a commercial pitch, so the licence of every dataset that ends up in the
merge matters. Creative Commons BY is fine as long as the source is credited. Anything marked
NonCommercial cannot be used here and has to be dropped, not quietly included.

kagglehub does not report licences, so check each dataset page on Kaggle and record what it
says below. Fill this in by hand.

In [ ]:
LICENCES = {
    "peru_mill": "TBD - check the Kaggle page",
    "philippines": "TBD - check the Kaggle page",
    "sethy": "TBD - original is Mendeley fwcj7stb8r, CC BY 4.0, attribution required",
    "dedeikhsan": "TBD - check the Kaggle page",
}

for k, v in LICENCES.items():
    print("%-14s %s" % (k, v))
print()
print("Any source marked NonCommercial must be removed from the source lists above.")

Step 1: Remove duplicate images

This is the step that decides whether the final accuracy number means anything, and it is the
easiest one to skip.

Several public rice leaf datasets are re-uploads of the same original. If one photo appears in
two sources, it can land in the training split and the test split at once. The model is then
marked on a question it has already been shown, and reports an accuracy it has not earned.

Two passes. The first removes files whose bytes are identical, which is always safe. The second
gives every image a short fingerprint based on what it looks like, so a re-saved or lightly
resized copy still matches, and removes images whose fingerprints are closer than a threshold.
That second pass is a judgement call, so it is counted separately and sample pairs are shown
afterwards for checking.

The fingerprint is taken from the middle of the image rather than the whole frame. A single rice
grain sits in the centre of a large plain background, and hashing the whole frame lets the
background drown out the grain, so unrelated grains look identical. An earlier version did
exactly that and threw away four fifths of the grain dataset.

The comparison runs across all sources pooled together, not one source at a time, because copies
between sources are exactly what we are looking for.

In [ ]:
HASH_SIZE = 16          # 16x16 -> 256 bits, enough detail to tell two grains apart
HASH_CROP = 0.7         # keep the middle 70 percent before hashing
MAX_DISTANCE = 3        # measured: genuine re-uploads land at 2 or less, distinct
                        # grains at 4 or more, so 3 separates them cleanly


def file_sha1(path, chunk=1 << 20):
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def dhash(path, size=HASH_SIZE, crop=HASH_CROP):
    """Difference hash: compares each pixel with its right neighbour. Survives resaving,
    resizing and mild compression, which is what a re-upload looks like.

    The centre crop matters. A single rice grain sits in the middle of a large plain
    background, and without cropping the background dominates the hash, so unrelated
    grains collide and get thrown away as duplicates."""
    with Image.open(path) as im:
        im = im.convert("L")
        if crop and crop < 1.0:
            w, h = im.size
            cw, ch = max(1, int(w * crop)), max(1, int(h * crop))
            left, top = (w - cw) // 2, (h - ch) // 2
            im = im.crop((left, top, left + cw, top + ch))
        im = im.resize((size + 1, size), Image.LANCZOS)
        a = np.asarray(im, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return np.packbits(bits)


_POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def hamming(row, rows):
    """Bit distance between one hash and many. Hashes are byte arrays now, not one
    integer, because 256 bits does not fit in a single machine word."""
    return _POPCOUNT[np.bitwise_xor(row, rows)].sum(axis=1)


def collect(root, source):
    out = []
    for sub in sorted(d for d in root.iterdir() if d.is_dir()):
        for f in sorted(sub.rglob("*")):
            if f.suffix.lower() in IMAGE_EXTS and f.is_file():
                out.append({"path": f, "raw_label": sub.name, "source": source})
    return out


def dedupe(records, max_distance=MAX_DISTANCE, max_examples=8):
    """Two passes. Identical bytes are always safe to drop. Visually near-identical is a
    judgement call, so it is counted separately and sample pairs are kept for inspection."""
    stats = {"before": len(records), "unreadable": 0, "exact_removed": 0,
             "perceptual_removed": 0, "pairs": collections.Counter(),
             "per_source_before": collections.Counter(),
             "per_source_removed": collections.Counter(), "examples": []}
    for r in records:
        stats["per_source_before"][r["source"]] += 1

    seen, staged = {}, []
    for r in records:
        try:
            sha = file_sha1(r["path"])
        except Exception:
            stats["unreadable"] += 1
            continue
        if sha in seen:
            stats["exact_removed"] += 1
            stats["per_source_removed"][r["source"]] += 1
            stats["pairs"][tuple(sorted((seen[sha]["source"], r["source"])))] += 1
            continue
        seen[sha] = r
        staged.append(r)

    hashes, kept = [], []
    for r in staged:
        try:
            r["hash"] = dhash(r["path"])
        except Exception:
            stats["unreadable"] += 1
            continue
        hashes.append(r["hash"])
        kept.append(r)

    if not kept:
        return [], stats

    arr = np.array(hashes, dtype=np.uint8)
    keep_mask = np.ones(len(kept), dtype=bool)

    for i in range(len(kept)):
        if not keep_mask[i]:
            continue
        rest = np.arange(i + 1, len(kept))
        rest = rest[keep_mask[rest]]
        if rest.size == 0:
            continue
        d = hamming(arr[i], arr[rest])
        for j, dist in zip(rest[d <= max_distance], d[d <= max_distance]):
            keep_mask[j] = False
            stats["perceptual_removed"] += 1
            stats["per_source_removed"][kept[j]["source"]] += 1
            stats["pairs"][tuple(sorted((kept[i]["source"], kept[j]["source"])))] += 1
            if len(stats["examples"]) < max_examples:
                stats["examples"].append((kept[i]["path"], kept[j]["path"], int(dist)))

    return [r for r, k in zip(kept, keep_mask) if k], stats

In [ ]:
ALARM_SHARE = 0.25      # losing more than a quarter of a source is a red flag, not a result


def report_dedupe(inv, title):
    records = []
    for name, (root, _counts) in inv.items():
        records += collect(root, name)
    survivors, s = dedupe(records)

    print(title)
    print("  images found        %6d" % s["before"])
    print("  unreadable          %6d" % s["unreadable"])
    print("  identical bytes     %6d" % s["exact_removed"])
    print("  visually identical  %6d" % s["perceptual_removed"])
    print("  remaining           %6d" % len(survivors))
    print()
    print("  removal rate by source:")
    alarms = []
    for src, n_before in sorted(s["per_source_before"].items()):
        n_removed = s["per_source_removed"][src]
        share = n_removed / n_before if n_before else 0
        flag = ""
        if share > ALARM_SHARE:
            flag = "   <- CHECK THIS"
            alarms.append((src, share))
        print("    %-14s %5d of %5d  %5.1f%%%s" % (src, n_removed, n_before, 100 * share, flag))
    if s["pairs"]:
        print()
        print("  duplicate pairs by source:")
        for (a, b), n in s["pairs"].most_common():
            tag = "within" if a == b else "ACROSS"
            print("    %-6s %-14s %-14s %5d" % (tag, a, b, n))
    if alarms:
        print()
        for src, share in alarms:
            print("  WARNING: %s lost %.0f%% of its images. Look at the sample pairs below."
                  % (src, 100 * share))
        print("  If those pairs are visibly different images, the threshold is wrong and the")
        print("  numbers from here on cannot be trusted. Do not carry on without checking.")
    print()
    return survivors, s


grain_records, grain_dedupe_stats = report_dedupe(grain_inv, "GRAIN")
leaf_records, leaf_dedupe_stats = report_dedupe(leaf_inv, "LEAF")

Look at the pairs below before going any further.

Each row is two images the notebook decided were the same. If they are obviously the same photo,
the threshold is right. If they are clearly two different grains or two different leaves, the
threshold is too loose, real training data is being thrown away, and every number after this
point is meaningless.

This check exists because an earlier run silently discarded eighty percent of the grain dataset
as duplicates when the images were not duplicates at all.

In [ ]:
def show_pairs(stats, title):
    import matplotlib.pyplot as plt

    ex = stats["examples"]
    if not ex:
        print(title + ": nothing flagged as a near-duplicate")
        return
    fig, axes = plt.subplots(len(ex), 2, figsize=(5, 2.4 * len(ex)))
    axes = np.atleast_2d(axes)
    for row, (kept_path, dropped_path, dist) in enumerate(ex):
        for col, p in enumerate((kept_path, dropped_path)):
            axes[row][col].imshow(Image.open(p))
            axes[row][col].axis("off")
            axes[row][col].set_title(("kept" if col == 0 else "dropped, distance %d" % dist),
                                     fontsize=8)
    fig.suptitle(title + " - flagged as the same image")
    plt.tight_layout()
    plt.show()


show_pairs(grain_dedupe_stats, "GRAIN")
show_pairs(leaf_dedupe_stats, "LEAF")

Step 2: Agree on one set of class names

The leaf sources label the same disease differently. One calls it Blast, another Rice Blast, a
third Leaf Blast. Left alone, the model would treat those as three separate diseases and split
its training data three ways.

The mapping below is written out in full, deliberately. Anything not in it is dropped and
counted, so an unrecognised label is visible rather than being quietly folded into whichever
class looks closest.

The final classes are the ones supported by more than one source and common in Philippine rice.
Diseases that appear in only one source with few images are dropped, because a class with fifty
examples will not learn and will drag down the whole confusion matrix.

In [ ]:
def norm(s):
    return "".join(c for c in s.lower() if c.isalnum())


LEAF_MAP = {
    "bacterialleafblight": "bacterial_leaf_blight",
    "bacterialblight": "bacterial_leaf_blight",
    "bacterial_leaf_blight": "bacterial_leaf_blight",
    "blb": "bacterial_leaf_blight",
    "riceblast": "rice_blast",
    "blast": "rice_blast",
    "leafblast": "rice_blast",
    "brownspot": "brown_spot",
    "tungro": "tungro",
    "tungrovirus": "tungro",
    "healthy": "healthy",
    "healthyriceplant": "healthy",
    "normal": "healthy",
}

GRAIN_MAP = {
    # the Peruvian dataset labels its folders in Spanish
    "entero": "whole",
    "quebrado": "broken",
    "tiza": "chalky",          # tiza is chalk
    "mancha": "stained",       # mancha is stain
    # english and other spellings, harmless if unused
    "whole": "whole",
    "broken": "broken",
    "partido": "broken",
    "chalky": "chalky",
    "yesoso": "chalky",
    "tizoso": "chalky",
    "stained": "stained",
    "manchado": "stained",
}

# Classes we are deliberately leaving out, each with too few images across too few sources
# to learn. Anything dropped that is NOT listed here stops the notebook.
LEAF_IGNORE = {"bakanae", "grassy_stunt_virus", "ragged_stunt_virus", "stem_rot",
               "rice_false_smut", "bacterial_leaf_streak", "sheath_rot",
               "narrow_brown_spot", "sheath_blight", "leaf_scald"}
GRAIN_IGNORE = set()

GRAIN_CLASSES = ["broken", "chalky", "stained", "whole"]
LEAF_CLASSES = ["bacterial_leaf_blight", "brown_spot", "healthy", "rice_blast", "tungro"]


def apply_map(records, mapping, title, ignore, expected):
    kept, dropped = [], collections.Counter()
    for r in records:
        canon = mapping.get(norm(r["raw_label"]))
        if canon is None:
            dropped[r["raw_label"]] += 1
            continue
        r["label"] = canon
        kept.append(r)

    print(title)
    counts = collections.Counter(r["label"] for r in kept)
    for k, v in sorted(counts.items(), key=lambda kv: -kv[1]):
        by_src = collections.Counter(r["source"] for r in kept if r["label"] == k)
        srcs = " ".join("%s=%d" % (s, n) for s, n in sorted(by_src.items()))
        print("  %-26s %5d   %s" % (k, v, srcs))
    print("  %-26s %5d" % ("TOTAL", len(kept)))
    if dropped:
        print("  dropped:")
        for k, v in dropped.most_common():
            tag = "on purpose" if norm(k) in {norm(x) for x in ignore} else "NOT RECOGNISED"
            print("    %-30s %5d   %s" % (k, v, tag))
    print()

    # An earlier run guessed two Spanish folder names wrongly, printed them here as dropped,
    # and trained a two-class model on data meant for four. A print was not enough.
    unexpected = {k: v for k, v in dropped.items() if norm(k) not in {norm(x) for x in ignore}}
    if unexpected:
        raise ValueError(
            "%s: unrecognised labels %s. Either add them to the mapping or list them in the "
            "ignore set to drop them on purpose." % (title, dict(unexpected)))

    if sorted(counts) != sorted(expected):
        raise ValueError("%s: got classes %s, expected %s. A folder was probably renamed."
                         % (title, sorted(counts), sorted(expected)))
    return kept


grain_records = apply_map(grain_records, GRAIN_MAP, "GRAIN after mapping",
                          GRAIN_IGNORE, GRAIN_CLASSES)
leaf_records = apply_map(leaf_records, LEAF_MAP, "LEAF after mapping",
                         LEAF_IGNORE, LEAF_CLASSES)

Every dropped label is marked either on purpose or NOT RECOGNISED, and anything not
recognised stops the notebook rather than letting it carry on with a smaller model than
intended. If it stops here, read the label it names. If it is a real class you want, add it to
the mapping. If you meant to leave it out, add it to the ignore set so the decision is written
down instead of implied.

The class list is also checked against the expected one, so a renamed folder in a future
version of a dataset fails loudly instead of quietly shrinking the model.

Step 3: Split into training, validation and test

Seventy percent to train on, fifteen to decide when to stop, fifteen held back.

The split is stratified, meaning each class is divided in the same proportions, so a rare class
still appears in all three. It is also seeded, so re-running gives the same split and two runs
can be compared.

The source each image came from is recorded alongside it. That is needed later, to check
whether the model has learned to recognise diseases or merely to recognise which dataset a
photo came from.

In [ ]:
def stratified_split(records, ratios=(0.70, 0.15, 0.15), seed=SEED):
    by_class = collections.defaultdict(list)
    for r in records:
        by_class[r["label"]].append(r)

    splits = {"train": [], "val": [], "test": []}
    rng = random.Random(seed)
    for label, items in sorted(by_class.items()):
        items = sorted(items, key=lambda r: str(r["path"]))
        rng.shuffle(items)
        n = len(items)
        n_train = int(n * ratios[0])
        n_val = int(n * ratios[1])
        # remainder goes to test, so nothing is lost to rounding
        splits["train"] += items[:n_train]
        splits["val"] += items[n_train:n_train + n_val]
        splits["test"] += items[n_train + n_val:]
    return splits


def materialise(splits, dest):
    """Ultralytics classification wants split/class folders on disk."""
    if dest.exists():
        shutil.rmtree(dest)
    manifest = []
    for split, items in splits.items():
        for i, r in enumerate(items):
            d = dest / split / r["label"]
            d.mkdir(parents=True, exist_ok=True)
            target = d / ("%s_%06d%s" % (r["source"], i, r["path"].suffix.lower()))
            shutil.copy2(r["path"], target)
            manifest.append({"split": split, "label": r["label"],
                             "source": r["source"], "file": str(target)})
    return manifest


def report_split(splits, title):
    print(title)
    labels = sorted({r["label"] for s in splits.values() for r in s})
    print("  %-26s %7s %7s %7s" % ("class", "train", "val", "test"))
    for lab in labels:
        row = [sum(1 for r in splits[s] if r["label"] == lab)
               for s in ("train", "val", "test")]
        flag = "   <- empty split" if 0 in row else ""
        print("  %-26s %7d %7d %7d%s" % (lab, row[0], row[1], row[2], flag))
    print("  %-26s %7d %7d %7d" % ("TOTAL", len(splits["train"]),
                                   len(splits["val"]), len(splits["test"])))
    print()


grain_splits = stratified_split(grain_records)
leaf_splits = stratified_split(leaf_records)
report_split(grain_splits, "GRAIN")
report_split(leaf_splits, "LEAF")

GRAIN_DIR = WORK / "grain"
LEAF_DIR = WORK / "leaf"
grain_manifest = materialise(grain_splits, GRAIN_DIR)
leaf_manifest = materialise(leaf_splits, LEAF_DIR)
json.dump(grain_manifest, open(OUT / "grain_manifest.json", "w"))
json.dump(leaf_manifest, open(OUT / "leaf_manifest.json", "w"))
print("written to", GRAIN_DIR, "and", LEAF_DIR)

No class should show an empty split. If one does, it has too few images to train on and
should be removed from the mapping in Step 2.

Step 4: Train

Two separate runs, one per model. Each starts from YOLOv11 weights already trained on
ImageNet, so it begins with a general sense of edges, texture and colour and only has to learn
what separates these particular classes. That is why a few thousand images is enough.

Training stops early when the validation score stops improving, so ending before the last
epoch is normal.

In [ ]:
EPOCHS = 60
IMGSZ = 224
PATIENCE = 10

def train(dataset_dir, name):
    model = YOLO("yolo11n-cls.pt")
    model.train(
        data=str(dataset_dir),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        patience=PATIENCE,
        seed=SEED,
        name=name,
        project=str(WORK / "runs"),
    )
    return model


grain_model = train(GRAIN_DIR, "grain_grade")

In [ ]:
leaf_model = train(LEAF_DIR, "leaf_disease")

Step 5: Measure it honestly

Everything up to here has been about earning the right to trust this number.

The test split is used now, once. Overall accuracy is reported, and then the part that matters
more: how the model does on each class separately, and what it confuses with what.

Precision is how often the model is right when it claims a class. Recall is how many of that
class it actually found. A class can have high precision and terrible recall, meaning the model
rarely claims it but is right when it does. Overall accuracy hides that completely.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt


def evaluate(model, dataset_dir, title):
    test_dir = dataset_dir / "test"
    classes = sorted(d.name for d in test_dir.iterdir() if d.is_dir())

    files, y_true = [], []
    for ci, c in enumerate(classes):
        for f in sorted((test_dir / c).iterdir()):
            if f.suffix.lower() in IMAGE_EXTS:
                files.append(str(f))
                y_true.append(ci)

    y_pred, confidences = [], []
    for i in range(0, len(files), 64):
        for res in model.predict(files[i:i + 64], imgsz=IMGSZ, verbose=False):
            y_pred.append(int(res.probs.top1))
            confidences.append(float(res.probs.top1conf))

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    acc = float((y_true == y_pred).mean())

    print("=" * 62)
    print(title)
    print("=" * 62)
    print("test images        %d" % len(files))
    print("top-1 accuracy     %.4f      <- the quotable number" % acc)
    print("mean confidence    %.4f" % float(np.mean(confidences)))
    print()
    print(classification_report(y_true, y_pred, target_names=classes, digits=3,
                                zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=range(len(classes)))
    fig, ax = plt.subplots(figsize=(1.1 * len(classes) + 3, 1.1 * len(classes) + 2))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(classes)), classes, rotation=45, ha="right")
    ax.set_yticks(range(len(classes)), classes)
    ax.set_xlabel("predicted")
    ax.set_ylabel("actual")
    ax.set_title(title + " - confusion matrix")
    for a in range(len(classes)):
        for b in range(len(classes)):
            ax.text(b, a, cm[a, b], ha="center", va="center",
                    color="white" if cm[a, b] > cm.max() / 2 else "black")
    plt.tight_layout()
    plt.show()

    return {"accuracy": acc, "classes": classes,
            "confusion_matrix": cm.tolist(),
            "report": classification_report(y_true, y_pred, target_names=classes,
                                            output_dict=True, zero_division=0)}


grain_metrics = evaluate(grain_model, GRAIN_DIR, "GRAIN GRADE")

In [ ]:
leaf_metrics = evaluate(leaf_model, LEAF_DIR, "LEAF DISEASE")

How to read what just printed.

A test accuracy far below the validation accuracy means the model memorised the training data
rather than learning from it.

A test accuracy above about 0.98 on merged public data is a reason to go back and check the
duplicate step, not a reason to celebrate. It usually means the same images appeared on both
sides.

Any class with recall below about 0.6 is a real weakness. Either it needs more images or it
should be dropped from the mapping.

In the confusion matrix, look at the off-diagonal cells. Two classes constantly mistaken for
each other is worth knowing about before someone at the mill finds it.

Does the leaf model know diseases, or does it know datasets?

The leaf data was merged from several sources photographed by different people with different
cameras and backgrounds. A model can score well by learning which dataset a photo came from and
guessing the disease most common in that dataset. It would look accurate here and fail
completely on a photo from a mill.

The check is to retrain with one source removed entirely, then test only on that source. If
accuracy holds up, the model has learned the disease. If it collapses, it had learned the
dataset.

This is a second training run, so it costs extra time. It is also the question a technical
reviewer is most likely to ask.

In [ ]:
def leave_one_out(records, held_out, epochs=25):
    train_recs = [r for r in records if r["source"] != held_out]
    test_recs = [r for r in records if r["source"] == held_out]
    if not test_recs or not train_recs:
        print("skipping %s: nothing to hold out" % held_out)
        return None

    labels_in_test = {r["label"] for r in test_recs}
    train_recs = [r for r in train_recs if r["label"] in labels_in_test]

    splits = stratified_split(train_recs, ratios=(0.85, 0.15, 0.0))
    splits["test"] = test_recs
    d = WORK / ("loo_" + held_out)
    materialise(splits, d)

    m = YOLO("yolo11n-cls.pt")
    m.train(data=str(d), epochs=epochs, imgsz=IMGSZ, patience=8, seed=SEED,
            name="loo_" + held_out, project=str(WORK / "runs"))
    return evaluate(m, d, "LEAF, trained without %s, tested only on %s" % (held_out, held_out))


leaf_sources_present = sorted({r["source"] for r in leaf_records})
print("sources available to hold out:", leaf_sources_present)
loo_metrics = {}
for src in leaf_sources_present:
    loo_metrics[src] = leave_one_out(leaf_records, src)

Collect everything

The run is not finished when training stops. The versions matter, because the server has to
match them, and the numbers matter because nobody can rebuild trust in a model whose accuracy
was never written down.

In [ ]:
!pip freeze > /content/artifacts/requirements-training.txt

import importlib.metadata as md_meta
import platform

versions = {}
for pkg in ["ultralytics", "torch", "torchvision", "numpy", "pillow", "scikit-learn"]:
    try:
        versions[pkg] = md_meta.version(pkg)
    except Exception:
        versions[pkg] = "not installed"

for name, model, metrics in (("grain_grade", grain_model, grain_metrics),
                             ("leaf_disease", leaf_model, leaf_metrics)):
    src = Path(model.trainer.save_dir) / "weights" / "best.pt"
    shutil.copy(src, OUT / (name + ".pt"))
    shutil.copytree(Path(model.trainer.save_dir), OUT / (name + "_run"),
                    dirs_exist_ok=True)
    print("saved %s.pt  (%.1f MB)" % (name, (OUT / (name + ".pt")).stat().st_size / 1e6))

facts = {
    "trained_on": platform.platform(),
    "gpu": torch.cuda.get_device_name(0),
    "seed": SEED,
    "split": "70/15/15 stratified, deduplicated before splitting",
    "grain": {"classes": grain_metrics["classes"],
              "test_accuracy": round(grain_metrics["accuracy"], 4),
              "class_names_from_model": grain_model.names},
    "leaf": {"classes": leaf_metrics["classes"],
             "test_accuracy": round(leaf_metrics["accuracy"], 4),
             "class_names_from_model": leaf_model.names},
    "licences": LICENCES,
    "versions": versions,
}

# The cross-dataset number is the one that predicts performance on a photo from the mill,
# so it belongs in the record rather than only on screen.
try:
    facts["leaf_cross_dataset"] = {
        src: (round(m["accuracy"], 4) if m else None)
        for src, m in loo_metrics.items()
    }
except NameError:
    facts["leaf_cross_dataset"] = "not run"
json.dump(facts, open(OUT / "model_facts.json", "w"), indent=2)

print()
print("=" * 62)
print("RECORD THESE")
print("=" * 62)
print("grain classes      ", grain_metrics["classes"])
print("grain test accuracy %.4f" % grain_metrics["accuracy"])
print("leaf classes       ", leaf_metrics["classes"])
print("leaf test accuracy  %.4f" % leaf_metrics["accuracy"])
if isinstance(facts["leaf_cross_dataset"], dict):
    print()
    print("leaf, tested on a source it never trained on:")
    for src, acc in sorted(facts["leaf_cross_dataset"].items()):
        print("  %-14s %s" % (src, "skipped" if acc is None else "%.4f" % acc))
    print("  ^ quote this one, not the number above")
print()
for k, v in versions.items():
    print("  %s==%s" % (k, v))

In [ ]:
shutil.make_archive("/content/rice_models", "zip", OUT)
print("rice_models.zip  %.1f MB" % (Path("/content/rice_models.zip").stat().st_size / 1e6))

from google.colab import files
files.download("/content/rice_models.zip")

Unpack the zip on your own machine and put the two weight files into the models folder of
the project.

grain_grade.pt
leaf_disease.pt

Keep a backup of the zip somewhere outside the project. The weight files are excluded from
version control because they are large, so nothing else is protecting them.

Keep requirements-training.txt as well. That is the version list the server has to match.

Then bring back the accuracy numbers and the class lists. Those class names become the keys the
API returns, so they have to be spelled exactly as printed.